In [ ]:
%load_ext autoreload
%autoreload 2
import sys, time
sys.path.insert(0, "..")
from pathlib import Path
import pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from src.experiments import log_experiment

DATA, MODELS = Path("../data/processed"), Path("../models")
MODELS.mkdir(exist_ok=True)

silver = pd.read_parquet(DATA / "silver_intent.parquet")
train, temp = train_test_split(silver, test_size=0.2, stratify=silver["intent"], random_state=42)
val, test = train_test_split(temp, test_size=0.5, stratify=temp["intent"], random_state=42)

pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])[
    ["tweet_id", "split"]].to_parquet(DATA / "ps1_splits.parquet", index=False)
print(f"train {len(train):,} | val {len(val):,} | test {len(test):,}")

train 25,734 | val 3,217 | test 3,217


In [2]:
def scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro")}

results = {}
for ngram in [(1, 1), (1, 2)]:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=ngram, min_df=3, max_df=0.9,
                                  sublinear_tf=True, max_features=50_000)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0)),
    ])
    t0 = time.time()
    pipe.fit(train["text_clean"], train["intent"])
    m = scores(val["intent"], pipe.predict(val["text_clean"]))
    results[ngram] = (pipe, m)
    log_experiment("PS-1", "TFIDF+LR", {"ngram": ngram, "C": 4.0, "min_df": 3, "eval": "silver_val"}, m)
    print(f"ngram={ngram}: acc {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} | {time.time() - t0:.0f}s")

best_ngram = max(results, key=lambda k: results[k][1]["macro_f1"])
best = results[best_ngram][0]
pred = best.predict(test["text_clean"])
print(f"\nBest: ngram={best_ngram} → silver TEST\n")
print(classification_report(test["intent"], pred, digits=3))

labels = sorted(test["intent"].unique())
print(pd.DataFrame(confusion_matrix(test["intent"], pred, labels=labels),
                   index=[f"true_{l}" for l in labels], columns=labels))

joblib.dump(best, MODELS / "ps1_tfidf_lr.joblib")

ngram=(1, 1): acc 0.9708 | macro-F1 0.9706 | 1s
ngram=(1, 2): acc 0.9745 | macro-F1 0.9749 | 1s

Best: ngram=(1, 2) → silver TEST

              precision    recall  f1-score   support

     Account      0.975     0.981     0.978       159
     Billing      0.983     0.957     0.970       487
   Complaint      0.957     0.979     0.968       682
     General      0.967     0.971     0.969      1115
   Technical      0.978     0.966     0.972       774

    accuracy                          0.970      3217
   macro avg      0.972     0.971     0.971      3217
weighted avg      0.970     0.970     0.970      3217

                Account  Billing  Complaint  General  Technical
true_Account        156        1          0        1          1
true_Billing          3      466          4       10          4
true_Complaint        0        1        668       12          1
true_General          0        5         16     1083         11
true_Technical        1        1         10       14        

['../models/ps1_tfidf_lr.joblib']

In [3]:
import numpy as np
import gensim.downloader as api
from gensim.models import Word2Vec

# Word2Vec corpus: all sample tweets except val / test / gold (no leakage)
all_tix = pd.read_parquet(DATA / "tickets_sample.parquet", columns=["tweet_id", "text_clean"])
gold_ids = pd.read_csv("../data/gold/gold_to_label.csv", usecols=["tweet_id"])["tweet_id"]
held_out = set(val["tweet_id"]) | set(test["tweet_id"]) | set(gold_ids)
corpus = [t.split() for t in all_tix.loc[~all_tix["tweet_id"].isin(held_out), "text_clean"]]

w2v = Word2Vec(corpus, vector_size=100, window=5, min_count=3, sg=1, workers=8, epochs=10, seed=42)
w2v.save(str(MODELS / "w2v_tickets.model"))
print(f"W2V vocab: {len(w2v.wv):,} | similar to 'refund':",
      [w for w, _ in w2v.wv.most_similar("refund", topn=5)])

glove = api.load("glove-twitter-100")

def mean_embed(texts, kv, dim=100):
    """Average the vectors of in-vocabulary words; zero vector if none are known."""
    X = np.zeros((len(texts), dim), dtype=np.float32)
    for i, t in enumerate(texts):
        vecs = [kv[w] for w in t.split() if w in kv]
        if vecs:
            X[i] = np.mean(vecs, axis=0)
    return X

for name, kv in [("Word2Vec (self-trained)", w2v.wv), ("GloVe-Twitter-100", glove)]:
    t0 = time.time()
    Xtr, Xva, Xte = (mean_embed(d["text_clean"].tolist(), kv) for d in (train, val, test))
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=4.0)
    clf.fit(Xtr, train["intent"])
    m_val = scores(val["intent"], clf.predict(Xva))
    m_test = scores(test["intent"], clf.predict(Xte))
    log_experiment("PS-1", f"{name}+LR", {"dim": 100, "pooling": "mean", "eval": "silver_val"}, m_val)
    print(f"{name}: val macro-F1 {m_val['macro_f1']:.4f} | test macro-F1 {m_test['macro_f1']:.4f} "
          f"| {time.time() - t0:.0f}s")
    joblib.dump(clf, MODELS / f"ps1_{name.split()[0].lower()}_lr.joblib")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


W2V vocab: 15,246 | similar to 'refund': ['partial', 'refunded', 'compensation', 'cheque', 'redelivery']
[==================================================] 100.0% 387.1/387.1MB downloaded
Word2Vec (self-trained): val macro-F1 0.8319 | test macro-F1 0.8244 | 1s
GloVe-Twitter-100: val macro-F1 0.7879 | test macro-F1 0.7829 | 1s


In [5]:
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(42)
MAX_TOKENS, SEQ_LEN, DIM = 20_000, 40, 100   # 40 tokens covers ~98% of tweets

le = LabelEncoder().fit(train["intent"])
ytr, yva, yte = (le.transform(d["intent"]) for d in (train, val, test))
as_arr = lambda d: np.array(d["text_clean"].tolist(), dtype=object)

vectorizer = layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN, standardize=None)
vectorizer.adapt(as_arr(train))
vocab = vectorizer.get_vocabulary()

# Embedding matrix: Word2Vec vector where known, small random otherwise; row 0 = padding
emb = np.random.normal(0, 0.1, (len(vocab), DIM)).astype("float32")
emb[0] = 0
known = [i for i, w in enumerate(vocab) if w in w2v.wv]
emb[known] = np.stack([w2v.wv[vocab[i]] for i in known])
print(f"Vocab {len(vocab):,} | Word2Vec init coverage {len(known) / len(vocab):.1%}")

Xtr, Xva, Xte = (vectorizer(as_arr(d)).numpy() for d in (train, val, test))

model = keras.Sequential([
    keras.Input(shape=(SEQ_LEN,), dtype="int64"),
    layers.Embedding(len(vocab), DIM, embeddings_initializer=keras.initializers.Constant(emb), mask_zero=True),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(le.classes_), activation="softmax"),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(le.classes_)), y=ytr)))
t0 = time.time()
model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=10, batch_size=128, class_weight=cw, verbose=2,
          callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)])

predict = lambda X: le.inverse_transform(model.predict(X, verbose=0).argmax(axis=1))
m_val, m_test = scores(val["intent"], predict(Xva)), scores(test["intent"], predict(Xte))
log_experiment("PS-1", "BiLSTM(W2V-init)", {"units": 64, "seq_len": SEQ_LEN, "dropout": 0.3,
                                            "eval": "silver_val"}, m_val)
print(f"\nBiLSTM: val macro-F1 {m_val['macro_f1']:.4f} | test macro-F1 {m_test['macro_f1']:.4f} | {time.time() - t0:.0f}s")

model.save(MODELS / "ps1_bilstm.keras")
joblib.dump(le, MODELS / "ps1_label_encoder.joblib")
(MODELS / "ps1_vocab.json").write_text(json.dumps(vocab))

Vocab 20,000 | Word2Vec init coverage 61.7%
Epoch 1/10
202/202 - 9s - 46ms/step - accuracy: 0.7429 - loss: 0.6217 - val_accuracy: 0.9136 - val_loss: 0.2647
Epoch 2/10
202/202 - 9s - 43ms/step - accuracy: 0.9435 - loss: 0.1676 - val_accuracy: 0.9531 - val_loss: 0.1384
Epoch 3/10
202/202 - 9s - 42ms/step - accuracy: 0.9754 - loss: 0.0754 - val_accuracy: 0.9608 - val_loss: 0.1287
Epoch 4/10
202/202 - 8s - 41ms/step - accuracy: 0.9861 - loss: 0.0439 - val_accuracy: 0.9639 - val_loss: 0.1248
Epoch 5/10
202/202 - 8s - 42ms/step - accuracy: 0.9910 - loss: 0.0298 - val_accuracy: 0.9671 - val_loss: 0.1187
Epoch 6/10
202/202 - 9s - 42ms/step - accuracy: 0.9920 - loss: 0.0279 - val_accuracy: 0.9723 - val_loss: 0.0904
Epoch 7/10
202/202 - 9s - 45ms/step - accuracy: 0.9944 - loss: 0.0191 - val_accuracy: 0.9745 - val_loss: 0.0879
Epoch 8/10
202/202 - 8s - 41ms/step - accuracy: 0.9952 - loss: 0.0147 - val_accuracy: 0.9677 - val_loss: 0.1173
Epoch 9/10
202/202 - 8s - 42ms/step - accuracy: 0.9977 - los

221753

In [8]:
from pathlib import Path
for folder in [Path("src"), Path("../src")]:
    print(f"\n{folder.resolve()}")
    if folder.exists():
        for f in sorted(folder.glob("*.py")):
            txt = f.read_text()
            print(f"  {f.name:16} {len(txt):>6} chars | INTENT_LFS: {'INTENT_LFS' in txt} | rule_priority: {'rule_priority' in txt}")
    else:
        print("  (does not exist)")


/Users/naren/Projects/Intelligent_Customer_Support_Ticket_System_Final_Project/Intelligent_Customer_Support_Ticket_System_Final_Project/notebooks/src
  (does not exist)

/Users/naren/Projects/Intelligent_Customer_Support_Ticket_System_Final_Project/Intelligent_Customer_Support_Ticket_System_Final_Project/src
  __init__.py           0 chars | INTENT_LFS: False | rule_priority: False
  cleaning.py         674 chars | INTENT_LFS: False | rule_priority: False
  experiments.py      746 chars | INTENT_LFS: False | rule_priority: False
  labeling.py        5328 chars | INTENT_LFS: True | rule_priority: True
